# Choice logit: model values for the psychometrics

**Question.** Meg's psychometrics use two values: the *true* magnitudes the task offers, and the
*believed* ones (what an arm paid the last time the mouse chose it). Neither is what the mouse
actually uses. This notebook estimates a third, **model value**, from the mouse's own choices, and
redraws the psychometrics and the stay/switch curves with it.

1. **The model.** A logistic regression of choice on port location (a fixed bias per tower),
   reward magnitude history and choice history.
2. **Synthetic data.** Simulated Q-learning mice, as in `scripts/to_integrate/RL model.ipynb`
   (same task simulator and value learner), whose real values and choice rule we know. The point:
   where the generating values are known, the model value recovers them.
3. **Real data.** The same pipeline on the drifting-task cohorts, next to the believed-value
   versions of the figures.

## 0. Setup and parameters

Everything below is a call into `src/` (`src/pipelines/choice_logit_values.py` and the modules it
uses); the notebook only chooses parameters, calls them, and explains the result.

Real data are loaded **cache-first** from the import cache (`BANDIT_CACHE_DIR`): the newest cached
snapshot of each cohort is used, nothing is written to the data share, and sessions added since
the snapshot are missing (a `StaleCacheWarning` says so). Set `DATA_ROOT` only if the data are not
under `BANDIT_DATA_ROOT`. Figures go to `results/` (git-ignored) as pdf, png and svg with a
`.caption.txt` beside each.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

REPO_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src/behavior_import").is_dir()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.agents.choice_logit_truth import q_kernel_bounds
from src.behavior_analysis import choice_logit_cv as cv
from src.behavior_analysis import model_value_psychometrics as mvp
from src.behavior_analysis.drifting_trial_table import drifting_trial_table
from src.behavior_analysis.get_choice_probs_by_value import count_choice_probs
from src.behavior_analysis.stay_switch_psychometrics import (
    stay_switch_curves_by_row,
    stay_switch_pairs,
)
from src.behavior_analysis.value_psychometrics import psychometric_curves
from src.behavior_visualization import plot_choice_logit as plots
from src.behavior_visualization.plot_stay_switch import plot_stay_switch_curves
from src.pipelines import choice_logit_values as pipe


def show(base):
    # figures are saved (pdf, png, svg, caption) by the plot functions; display the png here
    display(Image(filename=f"{base}.png"))

In [ ]:
# ---- parameters (edit here) ------------------------------------------------------------
COHORTS = [1, 3]  # cohorts whose drifting problems are pooled in the real-data section
DATA_ROOT = None  # None: BANDIT_DATA_ROOT. Loading is cache-first and never writes to the share.
OUTPUT_DIR = REPO_ROOT / "results" / "figures" / "choice-logit-values"

N_LAGS = 5  # reward lags and choice lags in the model
L2 = 1.0  # penalty on every weight (a Gaussian prior with SD 1 logit)
N_BINS = 5  # per-mouse quantile bins of the continuous value in the psychometrics
POOL_TRIALS = True  # within a mouse, pool the trials of a bin (False: average cells equally)

# Synthetic mice: (learning_rate, beta, kappa = stickiness, plus_bias = bias to plus-shaped ports)
PARAMETER_SETS = [
    (0.30, 1.0, 0.0, 0.0),  # plain Q-learner
    (0.30, 1.0, 1.0, 0.0),  # sticky
    (0.30, 1.0, 0.0, 0.8),  # tower (plus-port) bias
    (0.15, 1.2, 0.5, 0.5),  # slow learner, mild stickiness and bias
    (0.50, 0.8, 1.0, 0.5),  # fast learner, sticky and biased
]
MICE_PER_SET = 2
SYNTHETIC = dict(n_problems=5, sessions_per_problem=4, trials_per_session=200)
SEED = 20261106

synthetic_dir, real_dir = OUTPUT_DIR / "synthetic", OUTPUT_DIR / "real"

## 1. The model

On every trial the mouse chooses one of the three offered ports $a$. Each gets a utility

$$
U_a(t) \;=\; \underbrace{b_{\text{tower}(a)}}_{\text{port location}}
\;+\; \underbrace{\sum_{k=1}^{K} w^{r}_k\, r_a(t-k)}_{\text{reward history}}
\;+\; \underbrace{\sum_{k=1}^{K} w^{c}_k\, c_a(t-k)}_{\text{choice history}},
\qquad P(a)=\frac{e^{U_a}}{\sum_{a'} e^{U_{a'}}}
$$

- $r_a(t-k)$ is the reward magnitude (pulses, 0-4) the mouse **received** at port $a$ on trial $t-k$ if
  it chose $a$ then, otherwise 0.
- $c_a(t-k)$ is 1 if $a$ was chosen on trial $t-k$, otherwise 0. A positive $w^c_1$ is a tendency
  to repeat (perseveration).
- $b_{\text{tower}}$ is one fixed bias per tower (the 3x3 field has nine). Adding the same number to
  every tower's bias changes no probability, so the alphabetically first tower is fixed at 0.

It is a *conditional logit*: one set of weights per mouse, a softmax over the three offered ports,
fitted by maximum likelihood with a small L2 penalty (`L2`; a Gaussian prior with SD 1, weak next
to thousands of trials but enough to keep a feature that barely varies finite) and L-BFGS-B with
an exact gradient. A fit that does not converge raises.

**History rules.** Reward history carries over between the sessions of a problem and starts empty
at a problem's start, like the values of the Q-learner used in the RL notebook. Choice history is
cleared at a session start and after a dropped trial, as that learner's stickiness is. A lag that
reaches back before the start of the history is zero.

**Model value.** Only the reward part is called value:

$$\hat V_a(t) \;=\; \sum_{k=1}^{K} w^{r}_k\, r_a(t-k).$$

Perseveration (choice weights) and port preference (tower biases) are *nuisance* terms: they are in
$U$, so the fit does not credit the reward kernel for them, but they are left out of $\hat V$, so
they cannot show up as "value" in a psychometric curve.

**Why held-out.** $\hat V$ is used to explain the same choices it was fitted to, which would make
every psychometric look better than it is. So for each problem of a mouse the weights come from a
fit to the mouse's **other** problems (leave-one-problem-out, as in the RL notebook), and $\hat V$
on that problem is computed with them.

**Model comparison** (also held out): reward history only; + choice history; + tower bias (the full
model). Scores are negative log likelihood per choice in nats, against $\log 3 \approx 1.0986$
for choosing at random.

## 2. Synthetic data: Q-learning mice with known values

`simulate_cohort` makes agents of the value learner that the RL notebook and
`src/agents/drifting_simulation.py` use: a softmax over learned values $Q$ with inverse
temperature $\beta$, a stickiness bonus $\kappa$ on the previous choice, and a bias
`plus_bias` for the ports in the same row or column as the initiation port (this is the known
tower bias), after each choice $Q_a \leftarrow Q_a + \alpha\,(r - Q_a)$. They play the drifting
task (drift attempt rate 0.1, blocks of 30-35 trials, the most chosen arm drops to 0 at a block
change) on five different triples of towers, in sessions that go through the same importer code as
real data. The five parameter settings below are each run by two mice.

In [ ]:
synthetic = pipe.simulate_cohort(
    PARAMETER_SETS, mice_per_set=MICE_PER_SET, seed=SEED, **SYNTHETIC
)
print(f"{synthetic.trials['mouse'].nunique()} mice, {len(synthetic.trials):,} trials")
synthetic.parameters

### 2.1 Fit, and kernels against what the generating rule implies

The Q-learner is not exactly a lagged-reward logit, so "recovered" needs a precise target. A reward
that is followed by $m$ further visits to the same arm adds $\beta\alpha(1-\alpha)^m$ to that
arm's utility (an unchosen arm keeps its value, a chosen arm forgets by a factor $1-\alpha$ per
visit). For a reward $k$ trials back, $m$ lies between 0 (never chosen again) and $k-1$ (chosen on
every trial since), so the reward's contribution at lag $k$ lies between

$$\beta\alpha(1-\alpha)^{k-1}\quad\text{and}\quad\beta\alpha .$$

The table below gives these bounds as the scale to expect; the fitted weights need not fall inside them,
because the choice-history features share part of the effect. The reference line in the figure is sharper: the
least-squares weights of the logit's own features that best reproduce each agent's generating
utility $\beta Q + \kappa\,\mathrm{stay} + \text{plus\_bias}\cdot\mathrm{plus}$, i.e. what a logit would
find if the choices followed the generating rule exactly. Two things to expect:

- The reward kernel is flatter than a geometric decay and does not need to fall to zero within $K$ lags.
- The choice kernel is **not only stickiness**: stickiness $\kappa$ adds to choice lag 1, but a chosen arm's
  value is also pulled towards its newest reward, which makes older rewards of repeatedly chosen arms count
  less. A linear model can only express that through negative choice weights. The tower biases of the
  generating rule are $0$ or `plus_bias` for each tower (relative to the reference tower).

In [ ]:
analysis = pipe.analyse_trial_table(synthetic.trials, n_lags=N_LAGS, l2=L2)
reference = pipe.reference_weights(analysis, synthetic)
base = synthetic_dir / "Choice Logit Weights"
plots.plot_kernels(
    analysis.weights, base, reference=reference, reference_label="Generating Rule (Linear Summary)"
)
show(base)

# Implied bounds on the reward weight at each lag, next to the fitted and reference weights
bounds = pd.concat(
    {m: q_kernel_bounds(p, N_LAGS) for m, p in synthetic.parameters.set_index("mouse").iterrows()},
    names=["mouse"],
).reset_index(level=0)
reward = lambda w, name: (  # noqa: E731
    w[w["group"] == "reward"].rename(columns={"weight": name})[["mouse", "lag", name]]
)
bounds["lag"] = bounds["lag"].astype(float)
kernel_table = (
    bounds.merge(reward(analysis.weights, "fitted"), on=["mouse", "lag"])
    .merge(reward(reference, "reference"), on=["mouse", "lag"])
)
kernel_table.round(3).query("lag <= 3")

**Reading the figure.** Thin solid lines are the fitted weights of each mouse, dashed lines the reference of
the same mouse, the thick line their means. The reward kernels are positive and decay with lag,
the sticky mice have the largest positive choice-lag-1 weight, and in the right panel the fitted tower biases
fall on the identity line against the generating rule's (the plus-port bias is recovered).

### 2.2 Model comparison

Held-out negative log likelihood per choice: the full model should beat reward-only, and
choice history and tower bias should help exactly for the mice that have stickiness and bias.

In [ ]:
base = synthetic_dir / "Model Comparison"
plots.plot_model_comparison(analysis.comparison, base)
show(base)
gains = analysis.comparison.pivot(index="mouse", columns="model", values="gain_over_chance")[list(cv.MODELS)]
gains.join(synthetic.parameters.set_index("mouse")[["kappa", "plus_bias"]]).round(3)

### 2.3 Psychometrics: model value against the true values

For each trial, the three arms are ranked by value, and the psychometric curves show how often the mouse
chose the best and the second-ranked arm as a function of the best value, the second value, and their
difference. The integer true-value grid does not apply to a continuous $\hat V$, so each mouse's values
are cut into `N_BINS` **quantile bins** (equal numbers of that mouse's trials per bin, separately for each
x axis): value scales differ between mice, and equal-trial bins have comparable evidence everywhere.
A bin number is a rank among the mouse's trials, not a value. Ties (e.g. several zeros) share their rank
positions evenly, as in the other psychometrics.

Three rows, the same ranking of arms by:

1. the **model value** $\hat V$ (held out, from the logit),
2. the agents' **own Q-values** (the generating values, same quantile bins),
3. the **task's true magnitudes** (Meg's existing integer-grid psychometric).

If $\hat V$ recovers the generating values, row 1 should look like row 2.

In [ ]:
q_values = pipe.generating_values(analysis, synthetic)
model_curves = pipe.model_value_curves(analysis, N_BINS)
q_curves = pipe.model_value_curves(analysis, N_BINS, values=q_values)
task_counts = count_choice_probs(synthetic.problems, use_believed=False, tie_mode="split")
task_curves = psychometric_curves(task_counts, pool_trials=POOL_TRIALS, tie_mode="split")

rows = [
    plots.PsychometricRow("Model Value", model_curves, plots.value_labels("Model Value")),
    plots.PsychometricRow("Agent Q-Value", q_curves, plots.value_labels("Agent Q-Value")),
    plots.PsychometricRow("Task Value", task_curves, plots.value_labels("Task Value", binned=False)),
]
base = synthetic_dir / "Psychometric Curves"
plots.plot_psychometric_rows(
    rows,
    "Psychometric Curves: Model, Q and Task Values",
    (
        "Rows rank the three arms by the model value (held-out logit), by the simulated agents' own "
        "Q-value, and by the task's true magnitude.",
        f"Rows 1 and 2 use per-mouse quantile bins ({N_BINS}); row 3 is the integer-grid psychometric "
        "of the existing analysis (ties split).",
    ),
    base,
)
show(base)

**How close is $\hat V$ to the generating $Q$?** Within a trial only differences between arms matter,
so compare the trial-centred values (each arm minus the trial's mean over the three arms). Spearman
correlation of $\hat V$ with $Q$ per mouse:

In [ ]:
from scipy.stats import spearmanr

centred = lambda a: a - a.mean(axis=1, keepdims=True)  # noqa: E731
agreement = []
for mouse in analysis.designs:
    mine = (analysis.meta["mouse"] == mouse).to_numpy()
    rho = spearmanr(centred(q_values[mine]).ravel(), centred(analysis.values[mine]).ravel()).statistic
    agreement.append({"mouse": mouse, "spearman_vhat_vs_q": round(float(rho), 3)})
pd.DataFrame(agreement).set_index("mouse").T

### 2.4 Stay/switch curves on model values

P(stay) and P(switch) against the model value of the arm chosen on the previous trial and of its best
alternative. A continuous value has no "chose the 0-value option" row, so the rows are whether the previous
choice was **rewarded** (a non-zero magnitude) or not. Left to right: the difference
$V(\text{chosen}) - V(\text{alt})$, $V(\text{chosen})$ and $V(\text{alt})$, each in quantile bins. Below, the
same figure with the agents' own Q-values.

In [ ]:
pairs_model = pipe.model_value_pairs(analysis, N_BINS)
pairs_q = pipe.model_value_pairs(analysis, N_BINS, values=q_values)
for label, pairs in (("Model Value", pairs_model), ("Agent Q-Value", pairs_q)):
    curves = mvp.stay_switch_curves_by_reward(pairs, pool_trials=POOL_TRIALS)
    base = synthetic_dir / f"Stay-Switch Curves ({label})"
    plots.plot_stay_switch_by_reward(curves, label, POOL_TRIALS, base)
    show(base)

The model-value curves should show the same pattern as the Q-value ones: P(stay) rises with the value
of the chosen arm and falls with the value of the alternative. After an *unrewarded* choice the curves sit
lower (the mice switch more), and the sticky mice stay more in both rows. Where the model value is
less informative than the true $Q$ (early in a problem, or when the reward history has been forgotten),
the model-value curves are flatter.

## 3. Real data

The drifting problems of `COHORTS`, pooled as in the other all-cohort analyses (problems are keyed
`(cohort, number within the drifting type)`). The model is fitted per mouse and validated by leaving
one problem out. A mouse with a single drifting problem or fewer than 300 trials cannot be validated;
it is skipped with a warning and listed.

In [ ]:
problems = pipe.load_drifting_problems(COHORTS, DATA_ROOT)
trials = drifting_trial_table(problems)
trials.groupby(["mouse", "problem"]).size().unstack(fill_value=0)

In [ ]:
real = pipe.analyse_trial_table(trials, n_lags=N_LAGS, l2=L2)
print("skipped:", real.skipped or "none")
base = real_dir / "Choice Logit Weights"
plots.plot_kernels(real.weights, base)
show(base)

Weights are those of the full model fitted to all of a mouse's trials. Compare the reward kernel with
the Q-learner's from section 2: a positive, decaying kernel is what a learner that updates its values from
the rewards it receives looks like. The choice kernel shows perseveration (and, as explained above, part of
the correction for repeatedly chosen arms).

In [ ]:
base = real_dir / "Model Comparison"
plots.plot_model_comparison(real.comparison, base)
show(base)
real.comparison.pivot(index="mouse", columns="model", values="nats_per_choice")[list(cv.MODELS)].round(4)

### 3.1 Psychometrics: model value next to believed value

Row 1 ranks the arms by the held-out model value (quantile bins per mouse). Row 2 is the existing
*believed-value* psychometric (what each arm paid the last time the mouse chose it; ties split). The two
rows use different trial sets (believed values need all three arms to have been chosen once in the
problem, and have a limited integer grid), so they are compared by shape, not point by point.

In [ ]:
real_model_curves = pipe.model_value_curves(real, N_BINS)
believed_counts = count_choice_probs(problems, use_believed=True, tie_mode="split", warn_out_of_grid=False)
believed_curves = psychometric_curves(believed_counts, pool_trials=POOL_TRIALS, tie_mode="split")
rows = [
    plots.PsychometricRow("Model Value", real_model_curves, plots.value_labels("Model Value")),
    plots.PsychometricRow("Believed Value", believed_curves, plots.value_labels("Believed Value", binned=False)),
]
base = real_dir / "Psychometric Curves"
plots.plot_psychometric_rows(
    rows,
    "Psychometric Curves: Model and Believed Values",
    (
        "Row 1 ranks the arms by the held-out model value (per-mouse quantile bins); row 2 by the "
        "believed value (what an arm paid when last chosen; ties split; integer grid).",
        "Row 2 counts trials once all three arms of the problem have been chosen.",
    ),
    base,
)
show(base)

### 3.2 Stay/switch curves: model value next to believed value

First the model-value version (rows by whether the previous choice was rewarded), then the existing
believed-value version (rows by whether the chosen arm's believed value was 0).

In [ ]:
real_pairs = pipe.model_value_pairs(real, N_BINS)
curves = mvp.stay_switch_curves_by_reward(real_pairs, pool_trials=POOL_TRIALS)
base = real_dir / "Stay-Switch Curves (Model Value)"
plots.plot_stay_switch_by_reward(curves, "Model Value", POOL_TRIALS, base)
show(base)

believed_pairs = stay_switch_pairs(problems, use_believed=True, tie_mode="split")
believed_by_row = stay_switch_curves_by_row(believed_pairs.pairs, pool_trials=POOL_TRIALS)
base = real_dir / "Stay-Switch Curves (Believed Value)"
plot_stay_switch_curves(believed_by_row, "Believed", POOL_TRIALS, believed_pairs.ties, "split", base)
show(base)

### Notes and limits

- $\hat V$ is a *linear, finite-memory* summary of a mouse's reward history (K = `N_LAGS` trials), chosen
  because it can be read straight from choices. Section 2 shows how it differs from a Q-learner's
  value (flatter kernel; part of the Q-learner's forgetting sits in the choice weights); a real mouse need not be a Q-learner.
- The model value is in logits per pulse of its mouse. Quantile bins remove the scale but also the units:
  curves show shape, not size.
- Tower biases of towers that a fold's training problems never offered stay at 0 (the penalty), so
  the model comparison's tower step only helps where towers repeat across a mouse's problems.
- The believed-value figures use the cached sessions of the same cohorts, so they cover the same
  mice and problems as the model-value figures.